# XJTU rolling SOH and RUL forecasts

Estimate current SOH, predict SOH one, ten, and fifty cycles ahead, and refresh remaining useful life every five cycles from cycle 50. The latest completed discharge is from the preceding cycle. The current charge is available at each update.

Select the project Python kernel and run all cells. The notebook reads earlier CSV exports under reports and saves a comparison report and ZIP alongside them. All comparisons use the five saved cell-level folds. These settings were selected on this XJTU cohort.

The early model covers cycles 50 through 95. The established rolling model takes over at cycle 100. The exports support playback and separate benchmarks from cycle 50 or 100.

In [7]:
from collections import defaultdict
from contextlib import contextmanager
import csv
from io import TextIOWrapper
import json
from pathlib import Path, PurePosixPath
from statistics import mean, median
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

OUTPUT_ROOT = None  # Defaults to the project reports folder.

NB01 = "lifeguard_soh_results"
NB05 = "lifeguard_soh_error_audit_results"
NB06 = "lifeguard_rul_charge_features_results"
NB07 = "lifeguard_digital_twin_validation_results"
NB08 = "lifeguard_fixed_anchor_soh_results"
NB09 = "lifeguard_eol_timing_results"
NB10 = "lifeguard_average_and_worst_error_results"
OUTPUT_NAME = "lifeguard_rolling_forecasts_results"

EARLY_START = 50
ANCHOR = 100
STEP = 5
HORIZONS = (0, 1, 10, 50)
THRESHOLD = 80.0
DISTANCE_SCALE = np.array([50.0, 3.0, 0.04, 4.0])
NEIGHBOR_COUNT = 6
BOOSTING = dict(max_iter=60, max_leaf_nodes=9, min_samples_leaf=15, learning_rate=0.12,
                l2_regularization=1, monotonic_cst=[-1, 1, -1], early_stopping=False, random_state=0)

cwd = Path.cwd()
project = next((folder for folder in (cwd, *cwd.parents)
                if (folder / "README_LifeGuard_RL.md").is_file() and (folder / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open the notebook from the LifeGuard-RL project.")
reports = project / "reports"
output = (Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT is not None else reports) / OUTPUT_NAME


@contextmanager
def report_rows(report, table):
    folder = reports / report
    if (folder / table).is_file():
        with (folder / table).open(encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)
        return
    with ZipFile(folder.with_suffix(".zip")) as bundle:
        matches = [name for name in bundle.namelist() if PurePosixPath(name).name == table]
        if len(matches) != 1:
            raise ValueError(f"Expected one {table} in {report}.zip.")
        with TextIOWrapper(bundle.open(matches[0]), encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)


def at_anchor(row):
    return int(row["landmark_cycle"]) == ANCHOR


def unit(value):
    return min(1.0, max(0.0, value))

In [8]:
def load_saved_reports():
    with report_rows(NB08, "rollout_starts.csv") as rows:
        cells = {
            row["cell_id"]: {
                "rate": int(row["charge_rate_c"]), "fold": int(row["fold"]),
                "eol": int(row["observed_eol_cycle"]) if row["observed_eol_cycle"] else None,
                "fixed_rul": float(row["predicted_rul_cycles"]),
            }
            for row in rows if at_anchor(row) and row["initializer"] == "charge"
        }
    with report_rows(NB07, "cell_splits.csv") as rows:
        split = {(int(row["fold"]), row["role"], row["cell_id"]) for row in rows}
    folds = sorted({row["fold"] for row in cells.values()})
    expected = {(fold, "test" if fold == info["fold"] else "train", cell)
                for fold in folds for cell, info in cells.items()}
    if len(folds) != 5 or split != expected:
        raise ValueError("The Notebook 07 cell splits and Notebook 08 rollouts differ.")

    observed, fixed = defaultdict(dict), defaultdict(dict)
    with report_rows(NB08, "soh_trajectories.csv") as rows:
        for row in rows:
            if not at_anchor(row) or row["initializer"] != "charge":
                continue
            cell, cycle = row["cell_id"], int(row["cycle"])
            if int(row["fold"]) != cells[cell]["fold"]:
                raise ValueError(f"Wrong forecast fold for {cell} at cycle {cycle}.")
            observed[cell][cycle] = float(row["observed_soh_pct"])
            fixed[cell][cycle] = float(row["predicted_soh_pct"])

    charge = defaultdict(dict)
    with report_rows(NB05, "predictions.csv") as rows:
        for row in rows:
            if (row["design"] == "within_xjtu" and row["model"] == "first_cycle_calibrated"
                    and row["status"] == "current" and row["predicted_soh_pct"]):
                cell, cycle = row["cell_id"], int(row["cycle"])
                if int(row["fold"].rsplit("_", 1)[-1]) != cells[cell]["fold"]:
                    raise ValueError(f"Wrong SOH estimate fold for {cell}.")
                charge[cell][cycle] = float(row["predicted_soh_pct"])

    raw_charge = defaultdict(dict)
    with report_rows(NB01, "cycle_features.csv") as rows:
        for row in rows:
            cell, cycle = row["cell_id"], int(row["cycle"])
            if cycle in charge[cell]:
                raw_charge[cell][cycle] = float(row["prefix_charge_ah"]) * 50.0
    discharge = defaultdict(dict)
    with report_rows(NB01, "cycle_quality.csv") as rows:
        for row in rows:
            if row["soh_pct"] and row["status"] == "ok":
                discharge[row["cell_id"]][int(row["cycle"])] = float(row["soh_pct"])

    priors = defaultdict(dict)
    with report_rows(NB10, "rul_cell_predictions.csv") as rows:
        for row in rows:
            if at_anchor(row) and row["model"] in ("ridge_3c_recent_signals", "tail_penalized_linear"):
                cell = row["cell_id"]
                if (int(row["fold"]) != cells[cell]["fold"]
                        or int(row["observed_eol_cycle"]) != cells[cell]["eol"]):
                    raise ValueError(f"Notebook 10 RUL prior mismatch for {cell}.")
                priors[row["model"]][cell] = float(row["predicted_rul_cycles"])
    with report_rows(NB09, "cell_metrics.csv") as rows:
        for row in rows:
            if at_anchor(row) and row["model"] == "eol_timing_charge_discharge":
                cell = row["cell_id"]
                if (int(row["fold"]) != cells[cell]["fold"]
                        or int(row["observed_eol_cycle"]) != cells[cell]["eol"]):
                    raise ValueError(f"Notebook 09 RUL prior mismatch for {cell}.")
                priors[row["model"]][cell] = float(row["raw_rul_prediction_cycles"])

    for cell, data in cells.items():
        timeline = observed[cell]
        if ANCHOR not in timeline or (data["eol"] is not None and max(timeline) != data["eol"]):
            raise ValueError(f"Incomplete forecast timeline for {cell}.")
        for cycle, actual in timeline.items():
            if cycle in discharge[cell] and discharge[cell][cycle] != actual:
                raise ValueError(f"Measured SOH mismatch for {cell} at cycle {cycle}.")
    return cells, observed, fixed, charge, raw_charge, discharge, priors


cells, observed, fixed, charge, raw_charge, discharge, priors = load_saved_reports()
print(f"Loaded {len(cells)} cells, including {sum(x['eol'] is not None for x in cells.values())} verified EOLs.")

Loaded 23 cells, including 22 verified EOLs.


In [9]:
def recent_medians(history, last):
    window = [(cycle, history[cycle]) for cycle in range(last - 19, last + 1) if cycle in history]
    if len(window) < 10:
        return None
    early, recent = window[:5], window[-5:]
    gap = median([t for t, _ in recent]) - median([t for t, _ in early])
    first_level = median([value for _, value in early])
    recent_level = median([value for _, value in recent])
    return recent_level, (first_level - recent_level) / gap, gap


def checkpoint(cell, cycle, discharge_history, start):
    calibrated = charge[cell]
    if cycle not in calibrated:
        return None
    c_stats = recent_medians(calibrated, cycle)
    c_slope = max(c_stats[1], 0.0) if c_stats and c_stats[2] >= 10 else 0.0

    lag = cycle - 1
    d_stats = recent_medians(discharge_history, lag)
    last_soh = discharge_history.get(lag)
    d_slope = max(d_stats[1], 0.0) if d_stats and d_stats[2] >= 10 else 0.0
    d_vector = None
    if last_soh is not None and start - 1 in discharge_history and d_stats:
        d_vector = (cycle, last_soh, d_slope, discharge_history[start - 1])

    q_stats = recent_medians(raw_charge[cell], cycle)
    q_vector = None
    if cycle in raw_charge[cell] and start in raw_charge[cell] and q_stats:
        q_vector = (cycle, q_stats[0], q_stats[1], raw_charge[cell][start])
    return {
        "cell": cell, "cycle": cycle, "charge_soh": calibrated[cycle], "charge_slope": c_slope,
        "discharge": d_vector, "raw_charge": q_vector,
    }


def make_checkpoints(timeline, start):
    schedule = {cell: tuple(range(start, max(timeline[cell]), STEP)) for cell in cells}
    points = {
        cell: [item for cycle in schedule[cell]
               if (item := checkpoint(cell, cycle, discharge[cell], start)) is not None]
        for cell in cells
    }
    return schedule, points


def make_libraries(points):
    # Each training battery can contribute one neighbor per query.
    return {
        kind: {
            cell: (matches, np.asarray([point[kind] for point in matches]))
            for cell, group in points.items()
            if cells[cell]["eol"] is not None
            and (matches := [point for point in group if point[kind] is not None])
        }
        for kind in ("discharge", "raw_charge")
    }


scheduled, checkpoints = make_checkpoints(observed, ANCHOR)
libraries = make_libraries(checkpoints)


def nearest_training_points(point, kind, library):
    cell = point["cell"]
    current = cells[cell]
    query = point[kind]
    if query is None:
        return []
    neighbors = []
    for other, (points, vectors) in library[kind].items():
        info = cells[other]
        if info["fold"] == current["fold"] or info["rate"] != current["rate"]:
            continue
        distances = np.sum(((vectors - query) / DISTANCE_SCALE) ** 2, axis=1)
        index = int(np.argmin(distances))
        neighbors.append((float(distances[index]), other, points[index]))
    neighbors.sort(key=lambda item: (item[0], item[1]))
    return [(1 / (0.15 + dist), other, match) for dist, other, match in neighbors[:NEIGHBOR_COUNT]]


def weighted(values):
    weight = sum(w for w, _ in values)
    return sum(w * v for w, v in values) / weight if weight else None


def analog_rul(neighbors):
    return weighted([(w, cells[other]["eol"] - match["cycle"]) for w, other, match in neighbors])


def analog_soh(point, horizon, neighbors):
    changes = []
    for weight, other, match in neighbors:
        target = match["cycle"] + horizon
        if target > cells[other]["eol"]:
            outcome = THRESHOLD
        else:
            outcome = discharge[other].get(target)
        if outcome is not None:
            changes.append((weight, match["discharge"][1] - outcome))
    change = weighted(changes)
    return max(THRESHOLD, point["discharge"][1] - change) if change is not None else None


discharge_neighbors = {
    (point["cell"], point["cycle"]): nearest_training_points(point, "discharge", libraries)
    for group in checkpoints.values() for point in group
}
print(f"Prepared {sum(map(len, checkpoints.values()))} valid charge checkpoints.")

Prepared 937 valid charge checkpoints.


In [10]:
def charge_soh_forecast(point, horizon):
    rate = cells[point["cell"]]["rate"]
    current = point["charge_soh"]
    if horizon == 0:
        return current
    slope = 0 if horizon == 1 and rate == 2 else point["charge_slope"]
    return max(THRESHOLD, current - horizon * slope)


def selected_soh(point, horizon, neighbors, fixed_curves):
    cell, cycle = point["cell"], point["cycle"]
    rate = cells[cell]["rate"]
    earlier = charge_soh_forecast(point, horizon)
    if point["discharge"] is None:
        return fixed_curves[cell].get(cycle + horizon) if horizon == 50 else earlier, "charge_fallback"

    _, last_soh, slope, _ = point["discharge"]
    trend = max(THRESHOLD, last_soh - (horizon + 1) * slope)
    if horizon == 0:
        return (0.25 * earlier + 0.75 * trend if rate == 2 else trend), "past_discharge_trend"
    if horizon == 1 and rate == 3:
        return trend, "past_discharge_trend"
    if horizon == 10 and rate == 2:
        return earlier, "charge_trend"
    analog = analog_soh(point, horizon, neighbors)
    if analog is None:
        return (fixed_curves[cell].get(cycle + horizon) if horizon == 50 else earlier), "no_analog_fallback"
    if horizon == 1:
        return (analog + trend) / 2, "discharge_analog_and_trend"
    return analog, "discharge_analog"


def soh_row(point, horizon, estimate, method, timeline, fixed_curves):
    cell, origin = point["cell"], point["cycle"]
    target = origin + horizon
    actual = timeline[cell][target]
    fixed_forecast = fixed_curves[cell][target]
    earlier = fixed_forecast if horizon == 50 else charge_soh_forecast(point, horizon)
    return {
        "cell_id": cell, "fold": cells[cell]["fold"], "charge_rate_c": cells[cell]["rate"],
        "origin_cycle": origin, "target_cycle": target, "horizon_cycles": horizon,
        "method": method, "observed_soh_pct": actual, "fixed_soh_pct": fixed_forecast,
        "earlier_soh_pct": earlier, "selected_soh_pct": estimate,
        "fixed_abs_error_pp": abs(fixed_forecast - actual),
        "earlier_abs_error_pp": abs(earlier - actual),
        "selected_abs_error_pp": abs(estimate - actual),
    }


soh_predictions = []
for cell, group in checkpoints.items():
    for point in group:
        origin = point["cycle"]
        neighbors = discharge_neighbors[cell, origin]
        for horizon in HORIZONS:
            if origin + horizon in observed[cell]:
                estimate, method = selected_soh(point, horizon, neighbors, fixed)
                soh_predictions.append(soh_row(point, horizon, estimate, method, observed, fixed))

In [11]:
def initial_2c_rul():
    inputs = {}
    for cell, info in cells.items():
        if info["rate"] != 2:
            continue
        levels = [discharge[cell][cycle] for cycle in range(91, ANCHOR) if cycle in discharge[cell]]
        if len(levels) >= 5:
            inputs[cell] = median(levels)
    estimates = {}
    for cell, info in cells.items():
        if info["rate"] != 2:
            continue
        train = [other for other in sorted(inputs)
                 if cells[other]["fold"] != info["fold"] and cells[other]["eol"] is not None]
        if cell not in inputs or len(train) < 2:
            estimates[cell] = info["fixed_rul"]
            continue
        x = [[inputs[other]] for other in train]
        y = [cells[other]["eol"] - ANCHOR for other in train]
        estimator = make_pipeline(StandardScaler(), Ridge(alpha=3))
        estimator.fit(x, y)
        estimates[cell] = max(0.0, float(estimator.predict([[inputs[cell]]])[0]))
    return estimates


def late_3c_models():
    points = [point for cell, group in checkpoints.items()
              if cells[cell]["rate"] == 3 and cells[cell]["eol"] is not None
              for point in group if point["discharge"] is not None]
    models = {}
    for fold in sorted({row["fold"] for row in cells.values()}):
        training = [point for point in points if cells[point["cell"]]["fold"] != fold]
        counts = defaultdict(int)
        for point in training:
            counts[point["cell"]] += 1
        x = [point["discharge"][:3] for point in training]
        y = [cells[point["cell"]]["eol"] - point["cycle"] for point in training]
        weights = [len(training) / (len(counts) * counts[point["cell"]]) for point in training]
        model = HistGradientBoostingRegressor(**BOOSTING)
        model.fit(x, y, sample_weight=weights)
        models[fold] = model
    return models


def previous_3c_rul(point, fixed_remaining, discharge_match):
    cell, cycle = point["cell"], point["cycle"]
    raw = point["raw_charge"]
    if raw is None:
        return fixed_remaining
    _, raw_level, raw_slope, _ = raw
    raw_local = max(0.0, raw_level - THRESHOLD) / max(0.03, raw_slope)
    raw_weight = unit((88 - raw_level) / 5)
    raw_rolling = fixed_remaining + raw_weight * (raw_local - fixed_remaining)

    charge_peers = analog_rul(nearest_training_points(point, "raw_charge", libraries))
    if charge_peers is None:
        return raw_rolling
    elapsed = cycle - ANCHOR
    recent = max(0.0, priors["ridge_3c_recent_signals"][cell] - elapsed)
    peer_adjusted = max(0.0, recent + float(np.clip(charge_peers - recent, -60, 15)))
    baseline = raw_rolling + unit((raw_level - 88) / 3) * (peer_adjusted - raw_rolling)
    if point["discharge"] is None:
        return baseline

    _, last_soh, slope, _ = point["discharge"]
    # Persistent capacity and slow fade can justify raising an unusually short forecast.
    lower = max(0.0, last_soh - THRESHOLD) / max(0.18, slope + 0.08)
    baseline += 0.65 * unit((lower - 25) / 15) * max(0.0, lower - baseline)
    if discharge_match is not None:
        baseline -= unit(raw_level - last_soh) * max(0.0, baseline - discharge_match)

    # The mixed-input model includes discharge 100 and becomes eligible after that cycle.
    name = "eol_timing_charge_discharge" if cycle > ANCHOR else "tail_penalized_linear"
    early_reference = max(0.0, priors[name][cell] - elapsed)
    early_weight = (0.75 if cycle > ANCHOR else 0.5) * unit(1 - elapsed / 50) * unit((96 - last_soh) / 3)
    baseline -= early_weight * min(60.0, max(0.0, baseline - early_reference))

    upper = max(0.0, last_soh - THRESHOLD) / max(0.10, slope)
    late_weight = unit((92 - last_soh) / 4) * unit((slope - 0.04) / 0.05)
    return baseline - late_weight * max(0.0, baseline - upper)


def forecast_rul(point, peer_rul, anchors, boosters):
    cell, cycle = point["cell"], point["cycle"]
    info = cells[cell]
    elapsed = cycle - ANCHOR
    fixed_remaining = max(0.0, info["fixed_rul"] - elapsed)
    if info["rate"] == 2:
        revised = max(0.0, anchors[cell] - elapsed)
        if point["discharge"] is None or peer_rul is None:
            return fixed_remaining, fixed_remaining, revised, "2c_countdown"
        soh = point["discharge"][1]
        weight = 0.5 * unit((90 - soh) / 5)
        previous = fixed_remaining + weight * (peer_rul - fixed_remaining)
        return fixed_remaining, previous, revised + weight * (peer_rul - revised), "2c_ridge_and_analog"

    previous = previous_3c_rul(point, fixed_remaining, peer_rul)
    if point["discharge"] is None:
        return fixed_remaining, previous, previous, "3c_charge_fallback"
    features = point["discharge"][:3]
    boost = float(boosters[info["fold"]].predict([features])[0])
    weight = 0.75 * unit((94 - features[1]) / 6)
    updated = previous + weight * float(np.clip(boost - previous, -40, 40))
    return fixed_remaining, previous, max(0.0, updated), "3c_late_boosting" if weight else "3c_rolling"


def rul_row(cell, cycle, method, original, previous, current):
    info = cells[cell]
    actual = info["eol"] - cycle if info["eol"] is not None else None
    return {
        "cell_id": cell, "fold": info["fold"], "charge_rate_c": info["rate"], "cycle": cycle,
        "method": method, "observed_eol_cycle": info["eol"], "observed_rul_cycles": actual,
        "fixed_rul_cycles": original, "earlier_rul_cycles": previous, "selected_rul_cycles": current,
        "fixed_abs_error_cycles": abs(actual - original) if actual is not None else None,
        "earlier_abs_error_cycles": abs(actual - previous) if actual is not None else None,
        "selected_abs_error_cycles": abs(actual - current) if actual is not None else None,
    }


def rolling_rul_predictions():
    anchors = initial_2c_rul()
    boosters = late_3c_models()
    rows = []
    for cell, group in checkpoints.items():
        for point in group:
            cycle = point["cycle"]
            peer_rul = analog_rul(discharge_neighbors[cell, cycle])
            original, previous, current, method = forecast_rul(point, peer_rul, anchors, boosters)
            rows.append(rul_row(cell, cycle, method, original, previous, current))
    return rows, anchors, boosters


with threadpool_limits(limits=1):
    rul_predictions, anchors, boosters = rolling_rul_predictions()

## Rolling forecasts from cycle 50

Use charge signals through the origin cycle and discharge through the preceding cycle. An anchor estimate uses only the first 50 cycles of charge and the first 49 of discharge. Same-cycle peers and the anchor fits exclude the test battery's fold. Verified EOL is an outcome for training batteries and an evaluation label for the test battery. The early models use saved CSVs and keep the cycle-100 models on their original training timelines.

In [12]:
def load_early_references():
    measured, forecasts = defaultdict(dict), defaultdict(dict)
    durations = {}
    def is_early_reference(row):
        return (int(row["landmark_cycle"]) == EARLY_START and row["initializer"] == "charge"
                and row["model"] == "soh_dependent")

    with report_rows(NB07, "rollout_starts.csv") as rows:
        for row in rows:
            if is_early_reference(row):
                cell = row["cell_id"]
                eol = int(row["observed_eol_cycle"]) if row["observed_eol_cycle"] else None
                if int(row["fold"]) != cells[cell]["fold"] or eol != cells[cell]["eol"]:
                    raise ValueError(f"Cycle-50 rollout differs from the saved split and EOL for {cell}.")
                durations[cell] = float(row["predicted_rul_cycles"])

    with report_rows(NB07, "soh_trajectories.csv") as rows:
        for row in rows:
            if is_early_reference(row):
                cell, cycle = row["cell_id"], int(row["cycle"])
                if int(row["fold"]) != cells[cell]["fold"]:
                    raise ValueError(f"Wrong cycle-50 forecast fold for {cell}.")
                measured[cell][cycle] = float(row["observed_soh_pct"])
                forecasts[cell][cycle] = float(row["predicted_soh_pct"])

    references = {}
    with report_rows(NB06, "rul_predictions.csv") as rows:
        for row in rows:
            cell = row["cell_id"]
            reference = "protocol_median" if cells[cell]["rate"] == 2 else "ridge_3c_recent_signals_and_change"
            if int(row["landmark_cycle"]) == EARLY_START and row["model"] == reference:
                if int(row["fold"]) != cells[cell]["fold"]:
                    raise ValueError(f"Wrong cycle-50 RUL reference fold for {cell}.")
                references[cell] = EARLY_START + float(row["predicted_rul_cycles"])

    for cell, info in cells.items():
        timeline = measured[cell]
        if (cell not in durations or min(timeline, default=None) != EARLY_START
                or timeline.keys() != forecasts[cell].keys()
                or max(timeline) != max(observed[cell])
                or any(timeline.get(cycle) != value for cycle, value in observed[cell].items())):
            raise ValueError(f"Incomplete or misaligned cycle-50 timeline for {cell}.")
        if cell not in references:
            peers = [other["eol"] for other in cells.values()
                     if other["eol"] is not None and other["rate"] == info["rate"] and other["fold"] != info["fold"]]
            references[cell] = float(median(peers))
    return measured, forecasts, durations, references


EARLY_CHARGE_FIELDS = ("prefix_charge_ah", "rise_duration_3_8_to_4_0_s",
                       "rise_duration_4_0_to_4_1_s", "taper_duration_s", "taper_charge_ah")
SHAPE_CHARGE_FIELDS = (0, 3, 4)
PEER_CHARGE_FIELDS = slice(2, 4)  # Voltage rise from 4.0 to 4.1 V and taper duration.


def load_early_charge():
    signals = defaultdict(dict)
    with report_rows(NB01, "cycle_features.csv") as rows:
        for row in rows:
            cycle, cell = int(row["cycle"]), row["cell_id"]
            if cell in cells and cycle < ANCHOR:
                signals[cell][cycle] = [float(row[field]) if row[field] else np.nan
                                        for field in EARLY_CHARGE_FIELDS]
    return signals


def charge_window(history, first, last):
    values = [history[cycle] for cycle in range(first, last + 1) if cycle in history]
    if not values:
        return None
    values = np.asarray(values)
    finite = np.isfinite(values)
    if not finite.any(axis=0).all():
        return None
    return np.nanmedian(np.where(finite, values, np.nan), axis=0)


def discharge_window(cell, first, last):
    history = discharge[cell]
    values = [history[cycle] for cycle in range(first, last + 1) if cycle in history]
    return median(values) if values else None


def early_snapshot(cell, cycle, signals):
    recent_charge = charge_window(signals, cycle - 9, cycle)
    old_charge = charge_window(signals, cycle - 24, cycle - 15)
    recent_discharge = discharge_window(cell, cycle - 5, cycle - 1)
    old_discharge = discharge_window(cell, cycle - 20, cycle - 16)
    if any(value is None for value in (recent_charge, old_charge, recent_discharge, old_discharge)):
        return None
    return {
        "charge": recent_charge,
        "change": recent_charge - old_charge,
        "discharge": recent_discharge,
        "decline": old_discharge - recent_discharge,
    }


def anchor_shape_features(cell, signals):
    windows = [charge_window(signals, first, first + 9) for first in range(1, EARLY_START, 10)]
    if any(window is None for window in windows):
        return None
    tail = np.stack(windows)[:, SHAPE_CHARGE_FIELDS]
    recent = tail[-1]
    if cells[cell]["rate"] == 3:
        return np.r_[recent, (recent - tail[:-1]).ravel()], recent[1]
    recent_discharge = discharge_window(cell, 45, 49)
    mid_discharge = discharge_window(cell, 20, 29)
    last_discharge = discharge_window(cell, 40, 49)
    if any(value is None for value in (recent_discharge, mid_discharge, last_discharge)):
        return None
    mixed = np.r_[recent, (recent - tail[2:4]).ravel(), recent_discharge,
                  recent_discharge - mid_discharge, recent_discharge - last_discharge]
    charge_only = np.r_[recent, (recent - tail[1:4]).ravel()]
    return mixed, charge_only

In [13]:
def fit_ridge(x, y, query, alpha):
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(x, y)
    return float(model.predict([query])[0])


def fit_shape_ridge(x, y, query, alpha, log_target=False):
    x = np.asarray(x)
    center = x.mean(axis=0)
    # Near-constant window differences should stay small in the scaled input.
    scale = np.maximum(x.std(axis=0), 1e-8)
    model = Ridge(alpha=alpha)
    model.fit((x - center) / scale, np.log(y) if log_target else y)
    value = float(model.predict([(query - center) / scale])[0])
    return float(np.exp(value)) if log_target else value


def squared_distance(x, query):
    scale = np.maximum(x.std(axis=0), 1e-9)
    return np.sum(((x - query) / scale) ** 2, axis=1)


def early_training(cell, data, cycle=None):
    info = cells[cell]
    return [other for other, row in cells.items()
            if row["fold"] != info["fold"] and row["rate"] == info["rate"] and row["eol"] is not None
            and (other if cycle is None else (other, cycle)) in data]


def anchor_estimates(cell, snapshots, shape_features):
    if (cell, EARLY_START) not in snapshots:
        return None, None
    rate = cells[cell]["rate"]
    peers = early_training(cell, snapshots, EARLY_START)
    if len(peers) < 2:
        return None, None
    def initial_features(point):
        if rate == 2:
            return np.array([point["charge"][0], point["change"][0]])
        return np.r_[point["charge"], point["change"]]

    anchor = snapshots[cell, EARLY_START]
    targets = np.array([cells[other]["eol"] for other in peers])
    initial = fit_ridge([initial_features(snapshots[other, EARLY_START]) for other in peers],
                        targets, initial_features(anchor), 1)
    peers = early_training(cell, shape_features)
    if cell not in shape_features or len(peers) < 2:
        return initial, None
    y = np.array([cells[other]["eol"] for other in peers])
    a = fit_shape_ridge([shape_features[other][0] for other in peers], y, shape_features[cell][0],
                        1 if rate == 2 else 0.5, log_target=True)
    if rate == 2:
        b = fit_shape_ridge([shape_features[other][1] for other in peers], y, shape_features[cell][1], 0.5)
        return initial, 0.75 * a + 0.25 * b

    train = [[shape_features[other][1]] for other in peers]
    model = make_pipeline(StandardScaler(), KernelRidge(alpha=0.1, kernel="rbf", gamma=0.5))
    model.fit(train, y - y.mean())
    taper = float(y.mean() + model.predict([[shape_features[cell][1]]])[0])
    return initial, (a, taper)


def early_eol(cell, cycle, snapshots, anchor, reference):
    rate = cells[cell]["rate"]
    initial, shape = anchor
    value = snapshots.get((cell, cycle))
    peers = early_training(cell, snapshots, cycle)
    if value is None or initial is None or len(peers) < 2:
        return reference, "early_reference_fallback"

    y = np.array([cells[other]["eol"] for other in peers])
    if rate == 2:
        def features(point):
            return np.array([point["discharge"], point["charge"][0], point["change"][0]])
        update = fit_ridge(np.array([features(snapshots[other, cycle]) for other in peers]),
                           y, features(value), 1)
        prior = 0.5 * (initial + update)
        return ((0.5 * (prior + shape), "early_2c_ridge_ensemble") if shape is not None
                else (prior, "early_2c_ridge"))

    def features(point):
        return np.array([point["discharge"], *point["charge"][PEER_CHARGE_FIELDS]])
    x = np.array([features(snapshots[other, cycle]) for other in peers])
    distance = squared_distance(x, features(value))
    nearest = np.argsort(distance, kind="stable")[:5]
    update = np.average(y[nearest], weights=1 / (0.5 + distance[nearest]))
    prior = 0.75 * initial + 0.25 * update
    if shape is None:
        return prior, "early_3c_ridge_and_analog"
    a, taper = shape
    return 0.7 * (0.75 * prior + 0.25 * a) + 0.3 * taper, "early_3c_taper_ensemble"


EARLY_SOH_WEIGHTS = {
    (2, 0): (0.5, 0, 1.0, "trend"),
    (2, 1): (1.0, 0, 1.0, "trend"),
    (2, 10): (0.5, 0, 0.75, "trend"),
    (3, 0): (0.5, 4, 0.75, "trend"),
    (3, 1): (1.0, 0.5, 1.0, "trend"),
    (3, 10): (1.0, 8, 0.25, "rolling"),
}


def early_soh(point, horizon, eol, neighbors, snapshots, fixed_curves):
    cell, cycle = point["cell"], point["cycle"]
    rate = cells[cell]["rate"]
    reference, method = selected_soh(point, horizon, neighbors, fixed_curves)
    if point["discharge"] is None or (cell, cycle) not in snapshots:
        return reference, method
    if horizon == 50:
        if rate == 2 or not neighbors:
            return reference, method
        weight = 0.5 * unit((40 - (eol - (cycle + horizon))) / 40)
        return reference + weight * (THRESHOLD - reference), "early_3c_eol_adjusted_soh"

    def features(item):
        return np.array([item["discharge"], item["decline"], *item["charge"][PEER_CHARGE_FIELDS]])

    peers = [other for other in early_training(cell, snapshots, cycle)
             if cycle - 1 in discharge[other]
             and (cycle + horizon in discharge[other] or cycle + horizon > cells[other]["eol"])]
    if not peers:
        return reference, method
    x = np.array([features(snapshots[other, cycle]) for other in peers])
    sensor_weight, life_weight, blend, base = EARLY_SOH_WEIGHTS[rate, horizon]
    distance = sensor_weight * squared_distance(x, features(snapshots[cell, cycle]))
    lives = np.array([cells[other]["eol"] for other in peers])
    distance += life_weight * ((lives - eol) / 40) ** 2
    nearest = np.argsort(distance, kind="stable")[:NEIGHBOR_COUNT]
    deltas = [(discharge[other][cycle + horizon] if cycle + horizon <= cells[other]["eol"] else THRESHOLD)
              - discharge[other][cycle - 1] for other in peers]
    _, last_soh, slope, _ = point["discharge"]
    analog = max(THRESHOLD, last_soh + np.average(np.asarray(deltas)[nearest],
                                                  weights=1 / (0.5 + distance[nearest])))
    baseline = reference if base == "rolling" else max(THRESHOLD, last_soh - (horizon + 1) * slope)
    return blend * analog + (1 - blend) * baseline, "early_same_cycle_discharge_analog"


def rolling_early_predictions():
    measured, fixed_curves, durations, references = load_early_references()
    early_schedule, all_points = make_checkpoints(measured, EARLY_START)
    library = make_libraries(all_points)
    signals = load_early_charge()
    points = [point for group in all_points.values() for point in group if point["cycle"] < ANCHOR]
    snapshots = {(point["cell"], point["cycle"]): value for point in points
                 if (value := early_snapshot(point["cell"], point["cycle"], signals[point["cell"]])) is not None}
    shapes = {cell: value for cell in cells if (value := anchor_shape_features(cell, signals[cell])) is not None}
    estimates = {cell: anchor_estimates(cell, snapshots, shapes) for cell in cells}

    early_soh_rows, early_rul_rows = [], []
    for point in points:
        cell, cycle = point["cell"], point["cycle"]
        eol, method = early_eol(cell, cycle, snapshots, estimates[cell], references[cell])
        early_rul_rows.append(rul_row(cell, cycle, method,
                                      max(0.0, durations[cell] - (cycle - EARLY_START)),
                                      max(0.0, references[cell] - cycle), max(0.0, eol - cycle)))
        neighbors = nearest_training_points(point, "discharge", library)
        for horizon in HORIZONS:
            if cycle + horizon in measured[cell]:
                estimate, method = early_soh(point, horizon, eol, neighbors, snapshots, fixed_curves)
                early_soh_rows.append(soh_row(point, horizon, estimate, method, measured, fixed_curves))

    schedule = {cell: tuple(cycle for cycle in early_schedule[cell] if cycle < ANCHOR) + scheduled[cell]
                for cell in cells}
    return early_soh_rows, early_rul_rows, measured, schedule


with threadpool_limits(limits=1):
    early_soh_rows, early_rul_rows, timeline, combined_schedule = rolling_early_predictions()

soh_predictions = early_soh_rows + soh_predictions
rul_predictions = early_rul_rows + rul_predictions

In [14]:
def stats(rows, key, cycle_field):
    by_cell = defaultdict(list)
    for row in rows:
        by_cell[row["cell_id"]].append(row[key])
    worst = max(rows, key=lambda row: row[key]) if rows else None
    return {
        "cells": len(by_cell), "scored_points": len(rows),
        "mae": mean(mean(values) for values in by_cell.values()) if rows else None,
        "worst": worst[key] if worst is not None else None,
        "worst_cell": worst["cell_id"] if worst is not None else None,
        "worst_cycle": worst[cycle_field] if worst is not None else None,
    }


def metric_rows(filters, cycle_field, unit_label):
    groups, battery_metrics = [], []
    for context, rows, eligible in filters:
        for variant in ("fixed", "earlier", "selected"):
            key = f"{variant}_abs_error_{unit_label}"
            result = stats(rows, key, cycle_field)
            labels = {**context, "variant": variant}
            groups.append({
                **labels, "eligible_points": eligible,
                "coverage_pct": 100 * result["scored_points"] / eligible if eligible else None,
                **result,
            })
            by_cell = defaultdict(list)
            for row in rows:
                by_cell[row["cell_id"]].append(row)
            for cell, points in sorted(by_cell.items()):
                battery_metrics.append({**labels, "cell_id": cell, **stats(points, key, cycle_field)})
    return groups, battery_metrics


RUL_WINDOWS = ("all", "1–20", "21–50", ">50")
BENCHMARK_WINDOWS = ((EARLY_START, "early", ANCHOR), (EARLY_START, "all", None), (ANCHOR, "all", None))


def matches_rul_window(window, value):
    if window == "all":
        return True
    if window == "1–20":
        return 1 <= value <= 20
    if window == "21–50":
        return 21 <= value <= 50
    return value > 50


def benchmark_filters():
    soh, rul = [], []
    for start, window, stop in BENCHMARK_WINDOWS:
        def in_window(cycle):
            return cycle >= start and (stop is None or cycle < stop)

        for rate in (2, 3):
            cohort = [cell for cell in cells if cells[cell]["rate"] == rate]
            context = {"start_cycle": start, "origin_window": window, "charge_rate_c": rate}
            for horizon in HORIZONS:
                rows = [row for row in soh_predictions
                        if row["charge_rate_c"] == rate and row["horizon_cycles"] == horizon
                        and in_window(row["origin_cycle"])]
                eligible = sum(origin + horizon in timeline[cell] and in_window(origin)
                               for cell in cohort for origin in combined_schedule[cell])
                soh.append(({**context, "horizon_cycles": horizon}, rows, eligible))

            verified = [cell for cell in cohort if cells[cell]["eol"] is not None]
            remaining = [cells[cell]["eol"] - origin for cell in verified
                         for origin in combined_schedule[cell] if in_window(origin)]
            for rul_window in RUL_WINDOWS:
                rows = [row for row in rul_predictions
                        if row["charge_rate_c"] == rate and row["observed_rul_cycles"] is not None
                        and in_window(row["cycle"]) and matches_rul_window(rul_window, row["observed_rul_cycles"])]
                eligible = sum(matches_rul_window(rul_window, value) for value in remaining)
                rul.append(({**context, "actual_rul_window": rul_window}, rows, eligible))
    return soh, rul


soh_filters, rul_filters = benchmark_filters()
soh_metrics, soh_cell_metrics = metric_rows(soh_filters, "origin_cycle", "pp")
rul_metrics, rul_cell_metrics = metric_rows(rul_filters, "cycle", "cycles")


def result_summary():
    lines = [
        "# XJTU rolling SOH and RUL",
        "",
        "Each update uses the current charge and completed discharges through the preceding cycle. "
        "MAE weights each battery equally. SOH errors use percentage points and RUL errors use cycles.",
        "",
        "Earlier SOH uses the rolling charge-only estimate through ten cycles ahead and the saved "
        "fixed forecast at fifty cycles. The earlier RUL reference for cycles 50–95 uses the cycle-50 "
        "Notebook 06 timing model. The right-censored battery contributes to SOH but has no verified RUL label.",
        "",
    ]
    lookups = {
        "SOH": {(r["start_cycle"], r["origin_window"], r["charge_rate_c"], r["horizon_cycles"],
                 r["variant"]): r for r in soh_metrics},
        "RUL": {(r["start_cycle"], r["origin_window"], r["charge_rate_c"], r["actual_rul_window"],
                 r["variant"]): r for r in rul_metrics},
    }
    for start, window, stop in BENCHMARK_WINDOWS:
        heading = f"Updates at cycles {start}–{stop - STEP}" if stop is not None else f"All updates from cycle {start}"
        lines += [f"## {heading}", ""]
        for kind, categories in (("SOH", HORIZONS), ("RUL", RUL_WINDOWS)):
            label = "Horizon" if kind == "SOH" else "Actual cycles left"
            lines += [
                f"### {kind}", "",
                f"| Rate | {label} | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |",
                "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
            ]
            lookup = lookups[kind]
            for rate in (2, 3):
                for category in categories:
                    old, new = [lookup[start, window, rate, category, variant] for variant in ("earlier", "selected")]
                    if new["scored_points"]:
                        lines.append(f"| {rate}C | {category} | {new['scored_points']} | {new['coverage_pct']:.1f}% | "
                                     f"{old['mae']:.3f} | {new['mae']:.3f} | {old['worst']:.3f} | {new['worst']:.3f} | "
                                     f"{new['worst_cell']} at {new['worst_cycle']} |")
                    elif kind == "RUL" and category == "1–20":
                        lines.append(f"| {rate}C | {category} | 0 | — | — | — | — | — | — |")
            lines.append("")
    lines += [
        "SOH worst-error cycles refer to the origin of the forecast. Add the horizon to get the target cycle.",
        "RUL windows use verified observed EOL. The cycle-100 forecast model also exports current SOH at cycle 100.",
        "The early model settings were selected on this XJTU cohort. Performance on additional batteries is unverified.",
    ]
    return "\n".join(lines) + "\n"


tables = {
    "soh_predictions.csv": soh_predictions,
    "soh_metrics.csv": soh_metrics,
    "soh_cell_metrics.csv": soh_cell_metrics,
    "rul_predictions.csv": rul_predictions,
    "rul_metrics.csv": rul_metrics,
    "rul_cell_metrics.csv": rul_cell_metrics,
}
output.mkdir(parents=True, exist_ok=True)
for name, rows in tables.items():
    with (output / name).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

summary = result_summary()
(output / "summary.md").write_text(summary, encoding="utf-8")
config = {
    "start_cycle": EARLY_START, "start_cycles": [EARLY_START, ANCHOR],
    "checkpoint_interval": STEP, "soh_horizons": HORIZONS, "threshold_soh_pct": THRESHOLD,
    "neighbor_distance_scale": DISTANCE_SCALE.tolist(), "neighbor_count": NEIGHBOR_COUNT,
    "ridge_2c_alpha": 3, "boosting_3c": BOOSTING,
    "sources": [NB01, NB05, NB06, NB07, NB08, NB09, NB10],
}
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")

archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as bundle:
    for name in (*tables, "summary.md", "run_config.json"):
        bundle.write(output / name, name)

from IPython import get_ipython
from IPython.display import Markdown, display

if get_ipython() is not None:
    display(Markdown(summary))
else:
    print(summary)
print(f"Saved {archive_path}")

# XJTU rolling SOH and RUL

Each update uses the current charge and completed discharges through the preceding cycle. MAE weights each battery equally. SOH errors use percentage points and RUL errors use cycles.

Earlier SOH uses the rolling charge-only estimate through ten cycles ahead and the saved fixed forecast at fifty cycles. The earlier RUL reference for cycles 50–95 uses the cycle-50 Notebook 06 timing model. The right-censored battery contributes to SOH but has no verified RUL label.

## Updates at cycles 50–95

### SOH

| Rate | Horizon | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | 0 | 80 | 100.0% | 0.082 | 0.031 | 0.265 | 0.120 | XJTU_2C_battery-8 at 60 |
| 2C | 1 | 80 | 100.0% | 0.089 | 0.043 | 0.422 | 0.209 | XJTU_2C_battery-7 at 80 |
| 2C | 10 | 80 | 100.0% | 0.125 | 0.056 | 0.313 | 0.235 | XJTU_2C_battery-7 at 55 |
| 2C | 50 | 80 | 100.0% | 0.835 | 0.154 | 2.756 | 0.530 | XJTU_2C_battery-7 at 50 |
| 3C | 0 | 150 | 100.0% | 0.090 | 0.041 | 0.506 | 0.219 | XJTU_3C_battery-7 at 75 |
| 3C | 1 | 150 | 100.0% | 0.108 | 0.054 | 0.506 | 0.283 | XJTU_3C_battery-7 at 75 |
| 3C | 10 | 150 | 100.0% | 0.168 | 0.119 | 0.859 | 0.439 | XJTU_3C_battery-11 at 95 |
| 3C | 50 | 147 | 100.0% | 1.163 | 0.710 | 9.564 | 3.774 | XJTU_3C_battery-15 at 90 |

### RUL

| Rate | Actual cycles left | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | all | 70 | 100.0% | 9.000 | 6.995 | 17.000 | 13.439 | XJTU_2C_battery-1 at 55 |
| 2C | 1–20 | 0 | — | — | — | — | — | — |
| 2C | >50 | 70 | 100.0% | 9.000 | 6.995 | 17.000 | 13.439 | XJTU_2C_battery-1 at 55 |
| 3C | all | 150 | 100.0% | 20.449 | 13.464 | 56.985 | 32.419 | XJTU_3C_battery-9 at 50 |
| 3C | 1–20 | 0 | — | — | — | — | — | — |
| 3C | 21–50 | 4 | 100.0% | 16.260 | 1.860 | 16.260 | 2.196 | XJTU_3C_battery-11 at 95 |
| 3C | >50 | 146 | 100.0% | 20.449 | 13.451 | 56.985 | 32.419 | XJTU_3C_battery-9 at 50 |

## All updates from cycle 50

### SOH

| Rate | Horizon | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | 0 | 564 | 99.8% | 0.111 | 0.045 | 0.405 | 0.307 | XJTU_2C_battery-5 at 360 |
| 2C | 1 | 564 | 99.8% | 0.089 | 0.064 | 0.517 | 0.446 | XJTU_2C_battery-7 at 285 |
| 2C | 10 | 550 | 99.8% | 0.186 | 0.176 | 1.599 | 1.599 | XJTU_2C_battery-2 at 380 |
| 2C | 50 | 487 | 100.0% | 1.037 | 0.319 | 4.463 | 2.486 | XJTU_2C_battery-8 at 310 |
| 3C | 0 | 603 | 100.0% | 0.185 | 0.044 | 4.543 | 0.290 | XJTU_3C_battery-7 at 115 |
| 3C | 1 | 603 | 100.0% | 0.210 | 0.072 | 4.643 | 0.430 | XJTU_3C_battery-7 at 115 |
| 3C | 10 | 576 | 100.0% | 0.418 | 0.182 | 5.343 | 1.436 | XJTU_3C_battery-3 at 255 |
| 3C | 50 | 456 | 100.0% | 1.858 | 0.928 | 9.564 | 4.425 | XJTU_3C_battery-3 at 200 |

### RUL

| Rate | Actual cycles left | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | all | 495 | 99.8% | 8.179 | 7.357 | 17.000 | 14.915 | XJTU_2C_battery-3 at 100 |
| 2C | 1–20 | 27 | 96.4% | 5.259 | 4.918 | 16.795 | 14.175 | XJTU_2C_battery-1 at 385 |
| 2C | 21–50 | 42 | 100.0% | 4.944 | 5.187 | 14.868 | 12.249 | XJTU_2C_battery-1 at 365 |
| 2C | >50 | 426 | 100.0% | 8.679 | 7.718 | 17.000 | 14.915 | XJTU_2C_battery-3 at 100 |
| 3C | all | 603 | 100.0% | 15.067 | 12.644 | 56.985 | 37.615 | XJTU_3C_battery-9 at 100 |
| 3C | 1–20 | 60 | 100.0% | 3.907 | 2.391 | 15.305 | 8.848 | XJTU_3C_battery-3 at 265 |
| 3C | 21–50 | 90 | 100.0% | 9.303 | 6.832 | 24.823 | 20.463 | XJTU_3C_battery-8 at 220 |
| 3C | >50 | 453 | 100.0% | 17.718 | 15.059 | 56.985 | 37.615 | XJTU_3C_battery-9 at 100 |

## All updates from cycle 100

### SOH

| Rate | Horizon | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | 0 | 484 | 99.8% | 0.116 | 0.047 | 0.405 | 0.307 | XJTU_2C_battery-5 at 360 |
| 2C | 1 | 484 | 99.8% | 0.089 | 0.067 | 0.517 | 0.446 | XJTU_2C_battery-7 at 285 |
| 2C | 10 | 470 | 99.8% | 0.197 | 0.197 | 1.599 | 1.599 | XJTU_2C_battery-2 at 380 |
| 2C | 50 | 407 | 100.0% | 1.075 | 0.352 | 4.463 | 2.486 | XJTU_2C_battery-8 at 310 |
| 3C | 0 | 453 | 100.0% | 0.229 | 0.049 | 4.543 | 0.290 | XJTU_3C_battery-7 at 115 |
| 3C | 1 | 453 | 100.0% | 0.262 | 0.089 | 4.643 | 0.430 | XJTU_3C_battery-7 at 115 |
| 3C | 10 | 426 | 100.0% | 0.596 | 0.220 | 5.343 | 1.436 | XJTU_3C_battery-3 at 255 |
| 3C | 50 | 309 | 100.0% | 1.821 | 1.024 | 9.123 | 4.425 | XJTU_3C_battery-3 at 200 |

### RUL

| Rate | Actual cycles left | Paired points | Coverage | Earlier MAE | Selected MAE | Earlier max | Selected max | Worst selected at |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 2C | all | 425 | 99.8% | 8.045 | 7.404 | 17.000 | 14.915 | XJTU_2C_battery-3 at 100 |
| 2C | 1–20 | 27 | 96.4% | 5.259 | 4.918 | 16.795 | 14.175 | XJTU_2C_battery-1 at 385 |
| 2C | 21–50 | 42 | 100.0% | 4.944 | 5.187 | 14.868 | 12.249 | XJTU_2C_battery-1 at 365 |
| 2C | >50 | 356 | 100.0% | 8.616 | 7.841 | 17.000 | 14.915 | XJTU_2C_battery-3 at 100 |
| 3C | all | 453 | 100.0% | 13.051 | 12.187 | 37.615 | 37.615 | XJTU_3C_battery-9 at 100 |
| 3C | 1–20 | 60 | 100.0% | 3.907 | 2.391 | 15.305 | 8.848 | XJTU_3C_battery-3 at 265 |
| 3C | 21–50 | 86 | 100.0% | 8.894 | 7.156 | 24.823 | 20.463 | XJTU_3C_battery-8 at 220 |
| 3C | >50 | 307 | 100.0% | 16.818 | 16.666 | 37.615 | 37.615 | XJTU_3C_battery-9 at 100 |

SOH worst-error cycles refer to the origin of the forecast. Add the horizon to get the target cycle.
RUL windows use verified observed EOL. The cycle-100 forecast model also exports current SOH at cycle 100.
The early model settings were selected on this XJTU cohort. Performance on additional batteries is unverified.


Saved /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_rolling_forecasts_results.zip
